# Retrieval pilot bundle (Colab) — Manning-style Table 4 artefacts

**Purpose.** Produce reproducibility artefacts for **`knowledge_engine_vision.md`** Table 4: Appendix prompt freeze + hashing, retrieval run exports, **`lexical_tf_idf`**, optional **`dense_openai`**, judgment template + metrics, **`manifest.json`**, **`pip freeze`**.

**Columns.** (A) **Lexical TF‑IDF** on `combined_text` — always fills the manuscript lexical column when you run §6. (B) **OpenAI embeddings** — optional dense comparator; archive **`openai_doc_embeddings.npz`** for replay without re-calling the API. (C) **Vertex** — scaffold only until budget/infra returns.

**You still must:** load a corpus with stable `doc_id` + `combined_text`; set **`OPENAI_API_KEY`** in Colab secrets or env to run dense OpenAI; complete **`judgments.csv`**; archive the zipped folder (Zenodo) and paste DOI + git SHA into the manuscript.

**Not in Colab.** Ingestion git tag / commit — record in `META` manually.


## 1. Install & imports

In [ ]:
# Colab ships pandas/numpy; upgrading them can break numba/gradio/bqplot.
# Restart runtime after installs if you hit odd import errors.
!pip -q install --upgrade google-cloud-aiplatform scikit-learn openai


In [ ]:
import json
import hashlib
import subprocess
import pathlib
import random
import numpy as np
import pandas as pd

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

from datetime import datetime, timezone

RUN_ID = "pilot-draft"  # e.g. change to retrieval-pilot-2026-05-19

# Repro seeds (duplicate in manifest)
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

## 2. Output directory

Optional: **`from google.colab import drive; drive.mount('/content/drive')`** then set `OUTPUT_DIR` under Drive so the bundle survives Runtime disconnect.

In [ ]:
try:
    from google.colab import drive  # noqa: F401 — Colab-only
    COLAB_OR_LOCAL = "colab"
except ImportError:
    COLAB_OR_LOCAL = "local"

OUTPUT_DIR = pathlib.Path("/content/retrieval_pilot_bundle") / RUN_ID
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print("OUTPUT_DIR", OUTPUT_DIR, "environment", COLAB_OR_LOCAL)

## 3. Record pilot freeze identifiers (manual)

- **`EVAL_GIT_COMMIT`:** `git rev-parse HEAD` from the repo that freezes ingestion/API **or** `not-in-git-firestore-export` plus export checksum documented in manifest.
- **`VERTEX_*`:** model / index identifiers you actually queried (document only; omit secrets).
- Corpus: path or URI to the exported slice you judged (checksum below).

In [ ]:
META = {
    "run_id": RUN_ID,
    "utc_iso": datetime.now(timezone.utc).isoformat(),
    "runtime": COLAB_OR_LOCAL,
    "python_seed": SEED,
    "eval_git_commit": "REPLACE_WITH_git_rev_parse_HEAD",
    "corpus_description": "REPLACE e.g. Firestore/export-of-YYYYMMDD jsonl gz",
    "corpus_sha256": "REPLACE after you hash the frozen export file",
    "openai_embedding_model": "text-embedding-3-small",  # or text-embedding-3-large; document if changed
    "openai_embedding_dims": "REPLACE after first embed (e.g. 1536)",
    "vertex_embeddings_model": "REPLACE e.g. text-embedding-004",
    "vertex_index_or_endpoint": "REPLACE Matching Engine/index id",
    "eval_archive_doi": "REPLACE_AFTER_ZENODO_PUBLISH",
}
(OUTPUT_DIR / "meta_partial.json").write_text(json.dumps(META, indent=2))


### `pip freeze` for the archive README

In [ ]:
freeze = subprocess.check_output(
    ["pip", "freeze"], text=True
)
(OUTPUT_DIR / "requirements-frozen-on-colab.txt").write_text(freeze)

## 4. Appendix A — frozen prompt list + SHA (item 4)

**Must match manuscript order.** Excerpt mirrored from Appendix A (`knowledge_engine_vision.md`); extend if Appendix changes.

In [ ]:
APPENDIX_QUERIES = [
  {"id": 1, "text": "Probabilistic causal claims single-cell perturbation screens ML-heavy analyses summaries safeguards critiques contemporary arXiv since 2024"},
  {"id": 2, "text": "Contrasts transformer scaling-law narratives recurrent memory-augment architectures empirical ablations"},
  {"id": 3, "text": "Automated peer-review augmentation leakage critiques datasets"},
  {"id": 4, "text": "Dissipative adaptation thermodynamics juxtaposed maximum-entropy framings exploratory statistical physics neuroscience"},
  {"id": 5, "text": "Geometric-deep-learning biomedical graph anomaly open pitfalls"},
  {"id": 6, "text": "Uncertainty diffusion generative negatives survey"},
  {"id": 7, "text": "Jupyter-heavy neuroscience reproducibility container critiques"},
  {"id": 8, "text": "Synthetic chart vision-language benchmark distribution-shift critiques"},
  {"id": 9, "text": "Self-supervised protein representation pitfalls limited negative controls"},
  {"id": 10, "text": "Climate-econometrics causal inference disclaimers cross-domain vague exploratory robustness probe"},
  {"id": 11, "text": "Multilingual speech corpora governance/consent summarise"},
  {"id": 12, "text": "Robotics sim-to-real adversarial robustness surveys"},
  {"id": 13, "text": "Retrieval-augmented radiology assistants evaluation hallucination folklore verify responsibly"},
  {"id": 14, "text": "Graph neural oversmoothing critique positional-encoding remedies survey"},
  {"id": 15, "text": "Federated differential privacy genomics meta-analysis trade-offs summarise"},
  {"id": 16, "text": "Computational polarization modeling ethics exploratory caution"},
  {"id": 17, "text": "Transformer carbon-accounting methodological disputes summarise"},
  {"id": 18, "text": "Ontology alignment knowledge-graph embedding benchmarking contradictions survey"},
  {"id": 19, "text": "SOAR ACT-R CLARION versus neural hybrids history crosswalk summarise"},
  {"id": 20, "text": "Hallucination taxonomies conversational scientific assistants fabrication omission confabulation"},
  {"id": 21, "text": "Reading-comprehension benchmark saturation leaderboard inflation adversarial filtration"},
  {"id": 22, "text": "Astronomical anomaly-detection benchmarking dataset bias summarise"},
  {"id": 23, "text": "Multilingual toxicity classifier fairness cultural nuance critique"},
  {"id": 24, "text": "Wearable mental-health sensing informed-consent instrumentation critiques summarise"},
  {"id": 25, "text": "Differentiable physics simulators reproducibility pitfalls summarise"},
  {"id": 26, "text": "Dataset lineage combating scraped recycling summarise"},
  {"id": 27, "text": "Few-shot molecular property metric-learning scaffold-split pitfalls summarise"},
  {"id": 28, "text": "Conflicting adversarial purification empirical replications summarise cautiously"},
  {"id": 29, "text": "Large reasoning-model evaluation transparency chain-of-thought cautions summarise"},
  {"id": 30, "text": "Scientific foundation model benchmarking methodological governance cautions summarise"},
]


def sha256_bytes(b: bytes) -> str:
    return hashlib.sha256(b).hexdigest()


def write_json_stable(path: pathlib.Path, obj) -> None:
    path.write_text(json.dumps(obj, ensure_ascii=True, indent=2, sort_keys=True))


appendix_path = OUTPUT_DIR / "appendix_queries.json"
write_json_stable(appendix_path, {"queries": APPENDIX_QUERIES})
digest = sha256_bytes(appendix_path.read_bytes())
(OUTPUT_DIR / "appendix_queries.sha256").write_text(digest + "  appendix_queries.json\n")
print("appendix_queries.json sha256:", digest)
print("— paste as [INSERT_PROMPT_SNAPSHOT] or cite in README")

## 5. Load corpus (**you implement**)

**Contract.** `documents` rows: **`doc_id` (stable string)** and **`combined_text`** (title + abstract or passage text you legally index).\
Example sources: Drive parquet/CSV/jsonl upload, **`!gsutil cp`**, BigQuery dataframe.

In [ ]:
# --- Replace with YOUR loader ---
# documents = pd.read_parquet("/content/drive/MyDrive/...")

documents = pd.DataFrame([
    {"doc_id": "demo-001", "combined_text": "Example paper abstract about scaling laws transformers"},
    {"doc_id": "demo-002", "combined_text": "Example neuroscience reproducibility jupyter notebooks"},
])
assert {"doc_id", "combined_text"} <= set(documents.columns)
documents = documents.drop_duplicates(subset=["doc_id"]).reset_index(drop=True)

### Optional: hash corpus file after you freeze export

In [ ]:
# Example: META['corpus_sha256'] = sha256_bytes(pathlib.Path(CORPUS_PATH).read_bytes())

## 6. Lexical surrogate — TF-IDF pooled (Column A)

Matches manuscript wording at high level (“lexical surrogate TF-IDF pooled” on titles+abstracts you placed in **`combined_text`**).

In [ ]:
K = 10
vectorizer = TfidfVectorizer(
    lowercase=True,
    max_features=150_000,  # document if you tighten for scale
    dtype=np.float32,
)
corpus_mtx = vectorizer.fit_transform(documents["combined_text"])
doc_ids = documents["doc_id"].tolist()


def tfidf_topk(query_text: str) -> list[tuple[str, float]]:
    """Return up to top-K docs by cosine similarity; safe when corpus has fewer than K rows."""
    qv = vectorizer.transform([query_text])
    scores = cosine_similarity(qv, corpus_mtx)[0]
    n = int(scores.shape[0])
    if n == 0:
        return []
    k_eff = min(K, n)
    if k_eff == n:
        idx = np.argsort(scores)[::-1]
    else:
        idx = np.argpartition(scores, -k_eff)[-k_eff:]
        idx = idx[np.argsort(scores[idx])[::-1]]
    return [(doc_ids[i], float(scores[i])) for i in idx]


In [ ]:
def run_retriever(system_name: str, rank_fn):
    rows = []
    for q in APPENDIX_QUERIES:
        ranked = rank_fn(q["text"])
        for r, (doc_id, score) in enumerate(ranked, start=1):
            rows.append({
                "system": system_name,
                "query_id": q["id"],
                "rank": r,
                "doc_id": doc_id,
                "score": score,
            })
    return pd.DataFrame(rows)


rankings_lexical = run_retriever("lexical_tf_idf", tfidf_topk)
rankings_lexical.to_csv(OUTPUT_DIR / "rankings_lexical.csv", index=False)
rankings_lexical.head()


## 6b. OpenAI dense comparator (optional Column B)

**When to run.** Set environment variable **`OPENAI_API_KEY`** (Colab: *Secrets* or `os.environ`). If missing, the notebook skips dense ranking and writes an empty stub CSV.

**Contract.** Same `doc_id` order as `documents`; embed `combined_text` with **`OPENAI_EMBED_MODEL`**. Vectors are **L2-normalized**; scores are **dot product** (cosine for unit vectors). Large corpora: batched requests with **`OPENAI_BATCH_SIZE`**; **`openai_doc_embeddings.npz`** lets you **replay metrics** without re-embedding.

**Cost / time.** ~60k abstracts ⇒ many API calls — run once, then rely on saved `.npz` for Judgments + Table 4 iteration.


In [ ]:
import os
import time

OPENAI_EMBED_MODEL = os.environ.get("OPENAI_EMBED_MODEL", "text-embedding-3-small")
OPENAI_BATCH_SIZE = int(os.environ.get("OPENAI_BATCH_SIZE", "100"))
OPENAI_EMBED_MAX_CHARS = int(os.environ.get("OPENAI_EMBED_MAX_CHARS", "8000"))  # safety truncate per field

EMB_PATH = OUTPUT_DIR / "openai_doc_embeddings.npz"
RUN_OPENAI = bool(os.environ.get("OPENAI_API_KEY")) and len(documents) > 0

if RUN_OPENAI:
    from openai import OpenAI

    _client = OpenAI()

    def _normalize_rows(m: np.ndarray) -> np.ndarray:
        n = np.linalg.norm(m, axis=1, keepdims=True)
        n = np.maximum(n, 1e-12)
        return (m / n).astype(np.float32)

    def _embed_batch(texts: list[str]) -> np.ndarray:
        texts = [t[:OPENAI_EMBED_MAX_CHARS] if t else " " for t in texts]
        resp = _client.embeddings.create(model=OPENAI_EMBED_MODEL, input=texts)
        # API returns ordered `data` by index
        idx_map = {d.index: np.array(d.embedding, dtype=np.float32) for d in resp.data}
        return np.stack([idx_map[i] for i in range(len(texts))], axis=0)

    def embed_corpus_openai(texts: list[str]) -> np.ndarray:
        out = []
        for i in range(0, len(texts), OPENAI_BATCH_SIZE):
            batch = texts[i : i + OPENAI_BATCH_SIZE]
            out.append(_embed_batch(batch))
            time.sleep(0.05)
        return np.vstack(out).astype(np.float32)

    if EMB_PATH.exists():
        z = np.load(EMB_PATH, allow_pickle=True)
        openai_doc_ids = z["doc_ids"].tolist()
        openai_doc_emb = z["embeddings"].astype(np.float32)
        assert openai_doc_ids == documents["doc_id"].tolist(), "Cached embeddings doc_id order mismatch — delete .npz or reload documents"
        print("Loaded", EMB_PATH, openai_doc_emb.shape)
    else:
        openai_doc_emb = embed_corpus_openai(documents["combined_text"].astype(str).tolist())
        openai_doc_emb = _normalize_rows(openai_doc_emb)
        openai_doc_ids = documents["doc_id"].tolist()
        np.savez_compressed(
            EMB_PATH,
            doc_ids=np.array(openai_doc_ids, dtype=object),
            embeddings=openai_doc_emb,
            model=np.array(OPENAI_EMBED_MODEL),
        )
        print("Wrote", EMB_PATH, openai_doc_emb.shape)

    META["openai_embedding_model"] = OPENAI_EMBED_MODEL
    META["openai_embedding_dims"] = str(openai_doc_emb.shape[1])
else:
    print("Skipping OpenAI: set OPENAI_API_KEY to embed; or corpus empty")


In [ ]:
if RUN_OPENAI:

    def openai_dense_topk(query_text: str) -> list[tuple[str, float]]:
        qt = query_text[:OPENAI_EMBED_MAX_CHARS]
        qv = _embed_batch([qt])[0]
        qv = qv / max(float(np.linalg.norm(qv)), 1e-12)
        scores = openai_doc_emb @ qv
        n = scores.shape[0]
        k_eff = min(K, n)
        if k_eff == n:
            idx = np.argsort(scores)[::-1]
        else:
            idx = np.argpartition(scores, -k_eff)[-k_eff:]
            idx = idx[np.argsort(scores[idx])[::-1]]
        doc_ids = documents["doc_id"].tolist()
        return [(doc_ids[i], float(scores[i])) for i in idx]

    rankings_openai_dense = run_retriever("dense_openai", openai_dense_topk)
    rankings_openai_dense.to_csv(OUTPUT_DIR / "rankings_openai_dense.csv", index=False)
else:
    rankings_openai_dense = pd.DataFrame(columns=["system", "query_id", "rank", "doc_id", "score"])
    rankings_openai_dense.to_csv(OUTPUT_DIR / "rankings_openai_dense_STUB.csv", index=False)

(OUTPUT_DIR / "meta_partial.json").write_text(json.dumps(META, indent=2))
rankings_openai_dense.head() if RUN_OPENAI else rankings_openai_dense


## 7. Dense Vertex retrieval (Column B) — scaffold

**Fill in.** Either call your deployed Vertex Vector Search index, Matching Engine neighbours API, or re-rank embeddings you precomputed offline. Typical Colab login:

```python
from google.colab import auth
auth.authenticate_user()
# !gcloud config set project YOUR_PROJECT
```

Paste your batch embed + query loop here; **`rank_vertex_topk`** should return same shape as lexical.

In [ ]:
# def rank_vertex_topk(query_text: str) -> list[tuple[str, float]]:
#     """REPLACE: return top-K (doc_id, score) descending."""
#     raise NotImplementedError

# rankings_vertex = run_retriever("dense_vertex", rank_vertex_topk)
# rankings_vertex.to_csv(OUTPUT_DIR / "rankings_vertex.csv", index=False)

# Uncomment when implemented. For now workbook remains runnable with lexical+demo stubs.
rankings_vertex = pd.DataFrame(columns=["system","query_id","rank","doc_id","score"])
rankings_vertex.to_csv(OUTPUT_DIR / "rankings_vertex_STUB.csv", index=False)

## 8. Optional hybrid rerank (Column C) — scaffold

```python
def hybrid_topk(...): ...  # logistic / RRF blend — document formulation in README
```

## 9. Judgment kit (item 5)

**Note.** The default two-demo-paper corpus and auto-written demo **`judgments.csv`** (all **`relevant=0`**) make Table 4 metrics read as zeros on purpose. Replace **`documents`** and supply real **`judgments.csv`** before Springer.

**Template:** Union of **`doc_id`** appearing across systems’ top‑10 lists per **`query_id`**. Export CSV; annotate **`relevant`** as **`1`** or **`0`**. Re-import as **`judgments.csv`** with columns **`query_id`, `doc_id`, `relevant`**.

**Optional separate sheets:** replicate template per-system if judging each column independently.

In [ ]:
def union_judgment_template(rankings_dfs: list[pd.DataFrame]) -> pd.DataFrame:
    dfs = []
    for rdf in rankings_dfs:
        if rdf is None or rdf.empty:
            continue
        dfs.append(
            rdf[rdf["rank"] <= K][["query_id", "doc_id"]].drop_duplicates()
        )
    if not dfs:
        return pd.DataFrame(columns=["query_id", "doc_id", "relevant", "notes"])
    tmpl = pd.concat(dfs).drop_duplicates().sort_values(["query_id", "doc_id"])
    tmpl["relevant"] = ""
    tmpl["notes"] = ""
    return tmpl


# Defensive stubs if optional sections were skipped this session
if "rankings_openai_dense" not in globals():
    rankings_openai_dense = pd.DataFrame(
        columns=["system", "query_id", "rank", "doc_id", "score"],
    )
if "rankings_vertex" not in globals():
    rankings_vertex = pd.DataFrame(
        columns=["system", "query_id", "rank", "doc_id", "score"],
    )

_rank_dfs = [rankings_lexical]
if rankings_openai_dense is not None and not rankings_openai_dense.empty:
    _rank_dfs.append(rankings_openai_dense)
if rankings_vertex is not None and not rankings_vertex.empty:
    _rank_dfs.append(rankings_vertex)
_tmpl = union_judgment_template(_rank_dfs)
_tmpl.to_csv(OUTPUT_DIR / "judgments_TEMPLATE.csv", index=False)
_tmpl.head()

## 10. Metrics for Table 4 (item 2) — reads `judgments.csv`

In [ ]:

def reciprocal_rank(rank_list: np.ndarray, rel: np.ndarray) -> float:
    """First relevant rank (binary rel), 1-indexed reciprocal; 0 if none."""
    for idx, (_, is_rel) in enumerate(zip(rank_list, rel), start=1):
        if is_rel >= 1:
            return 1.0 / idx
    return 0.0


def dcg_binary(rel: np.ndarray) -> float:
    gains = np.asarray(rel[:K], dtype=np.float64)
    positions = np.arange(1, len(gains) + 1)
    return float(np.sum((2**gains - 1) / np.log2(positions + 1)))


def ndcg_binary(rel_ranked: np.ndarray) -> float:
    rel_sorted = np.sort(rel_ranked)[::-1][:K]
    ideal = np.sort(rel_sorted)[::-1]
    denom = dcg_binary(ideal)
    if denom <= 0:
        return 0.0
    return dcg_binary(rel_ranked[:K]) / denom


def summarise_system(rank_df: pd.DataFrame, judgments: pd.DataFrame) -> dict:
    j = judgments.astype({"query_id": int})
    ndcgs = []
    precs = []
    rrs = []
    missing = []
    rel_map = j.set_index(["query_id", "doc_id"])["relevant"]
    rel_map = rel_map.astype(int)

    for qid, sub in rank_df.groupby("query_id"):
        sub = sub.sort_values("rank")
        docs = sub["doc_id"].tolist()[:K]
        rel_vec = []
        for d in docs:
            try:
                rel_vec.append(rel_map.loc[(qid, d)])
            except KeyError as exc:
                raise KeyError(
                    "Missing judgment for",
                    qid,
                    d,
                    "complete judgments.csv or widen template",
                ) from exc

        rel_vec = np.asarray(rel_vec, dtype=np.int32)

        ndcgs.append(ndcg_binary(rel_vec))
        precs.append(float(rel_vec.mean()))  # Precision@10 = relevant-in-top-k / K
        rr = reciprocal_rank(np.arange(K), rel_vec)
        rrs.append(rr)
        missing.append(1 if rel_vec.sum() == 0 else 0)

    return {
        "mean_ndcg_at_10": float(np.mean(ndcgs)),
        "mean_precision_at_10": float(np.mean(precs)),
        "median_reciprocal_rank": float(np.median(rrs)),
        "queries_no_relevant_in_top10": int(np.sum(missing)),
        "n_queries": len(ndcgs),
    }


# DEMO: judgments all zeros breaks interpretability — supply real judgments before publication
judgments_path = OUTPUT_DIR / "judgments.csv"
if not judgments_path.exists():
    small = []
    for row in rankings_lexical.itertuples():
        small.append(
            {"query_id": row.query_id, "doc_id": row.doc_id, "relevant": 0},
        )
    demo = (
        pd.DataFrame(small).drop_duplicates(subset=["query_id", "doc_id"]).reset_index(drop=True)
    )
    demo.to_csv(judgments_path, index=False)
    print("Wrote DEMO judgments.csv", judgments_path)

_judgments = pd.read_csv(judgments_path)
assert set(_judgments.columns) >= {"query_id", "doc_id", "relevant"}
_tab = summarise_system(rankings_lexical, _judgments)
print("Lexical metrics (sanity DEMO):", _tab)

In [ ]:
def metric_table_lexical(rankings_lexical_df, judgments):
    """Aligns with manuscript Table 4 single lexical column dense-vector abstaining."""
    tbl = []
    metric_order = [
        "mean_ndcg_at_10",
        "mean_precision_at_10",
        "median_reciprocal_rank",
        "queries_no_relevant_in_top10",
    ]
    s = summarise_system(rankings_lexical_df, judgments)
    for m in metric_order:
        tbl.append({"Manuscript_metric": m, "value": s[m]})
    return pd.DataFrame(tbl)

# Run after judgments.csv populated (overwrite demo zeros)
tbl4 = metric_table_lexical(rankings_lexical, _judgments)
tbl4.to_csv(OUTPUT_DIR / "table4_metrics_lexical_long.csv", index=False)
tbl4["Table4_row"] = [
    "Mean nDCG@10",
    "Mean Precision@10",
    "Median reciprocal rank",
    "Queries lacking relevant passage in top-10",
]
print(tbl4[["Table4_row", "value"]])
# Two-column Table 4 export (lexical vs OpenAI dense) — manuscript may use both
_tab_o = summarise_system(rankings_openai_dense, _judgments) if RUN_OPENAI and not rankings_openai_dense.empty else None
if _tab_o:
    rows = []
    for label, key in [
        ("Mean nDCG@10", "mean_ndcg_at_10"),
        ("Mean Precision@10", "mean_precision_at_10"),
        ("Median reciprocal rank", "median_reciprocal_rank"),
        ("Queries lacking relevant passage in top-10", "queries_no_relevant_in_top10"),
    ]:
        rows.append({"metric": label, "lexical_tf_idf": _tab[key], "dense_openai": _tab_o[key]})
    pd.DataFrame(rows).to_csv(OUTPUT_DIR / "table4_metrics_lexical_vs_openai.csv", index=False)
    print(pd.DataFrame(rows))



## 11. `manifest.json` + zenodo-ready zip checklist (item 1)

1. Update **`META`** cell with commit + corpus checksum appendix sha256 (`vertex_*` blank when abstaining) + DOI when published.
2. Include: this notebook (**File → Save a copy → Download .ipynb**), `appendix_queries.json`, `requirements-frozen-on-colab.txt`, all `rankings_*.csv`, `judgments.csv`, `judgments_TEMPLATE.csv`, `README.md` prose you paste from Colab markdown cell.
3. Zip → Zenodo draft → cite DOI (`[INSERT_EVAL_ARCHIVE_DOI]`).

In [ ]:
manifest = {
    **META,
    "appendix_queries_sha256": digest,
    "files": [
        "appendix_queries.sha256",
        "appendix_queries.json",
        "meta_partial.json",
        "requirements-frozen-on-colab.txt",
        "rankings_lexical.csv",
        "judgments_TEMPLATE.csv",
        "judgments.csv",
        "table4_metrics_lexical_long.csv",
    ],
}
(OUTPUT_DIR / "manifest.json").write_text(json.dumps(manifest, indent=2))

### Download bundle from Colab

```python
import shutil
shutil.make_archive(f"/content/{RUN_ID}_bundle", "zip", OUTPUT_DIR.parent, OUTPUT_DIR.name)
from google.colab import files
files.download(f"/content/{RUN_ID}_bundle.zip")
```